# 2.3 多普勒频移与差分相移计算

**学习目标**
- 掌握多普勒频移与径向速度的定量关系
- 理解H和V通道之间差分相移的产生机制
- 计算不同波长下的多普勒频移和差分相移
- 理解KDP（比差分相移）的物理意义

**参考文献**：Ryzhkov & Zrnic, Chapter 2, pp. 196-230

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets
plt.rcParams['font.family'] = ['DejaVu Sans', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 1. 理论背景

### 多普勒频移

当雷达波束与目标有相对径向运动时，接收频率发生偏移：

$$f_D = \frac{2v_r}{\lambda}$$

其中 $v_r$ 是径向速度（远离为正），$\lambda$ 是雷达波长。

### 差分相移 ΦDP

H和V偏振波在介质中传播时由于相位速度不同而产生相位差：

$$\Phi_{DP} = \frac{2\pi}{\lambda}\int_0^R [K_h(s) - K_v(s)] ds = \int_0^R K_{DP}(s) ds$$

其中 $K_{DP}$ 是比差分相移（单位：°/km 或 rad/km）。

In [ ]:
def calculate_doppler_frequency(velocity, wavelength):
    """计算多普勒频移"""
    return 2 * velocity / wavelength

def calculate_differential_phase(KDP, distance):
    """计算差分相移"""
    return KDP * distance

def calculate_radial_velocity(f_d, wavelength):
    """从多普勒频移计算径向速度"""
    return f_d * wavelength / 2

def kdp_from_rainrate(R, wavelength):
    """
    从降雨率估计KDP（经验公式）
    R: 降雨率 mm/h
    wavelength: m
    """
    if wavelength > 0.08:  # S-band
        return 0.014 * R**0.95
    elif wavelength > 0.04:  # C-band
        return 0.028 * R**0.93
    else:  # X-band
        return 0.062 * R**0.92

## 2. 多普勒频移与波长的关系

In [ ]:
def plot_doppler_calculator():
    """多普勒频移计算器"""
    
    # 速度范围
    v_range = np.linspace(-100, 100, 500)
    
    # 不同波段的波长
    wavelengths = {'X-band (3.2 cm)': 0.032, 
                   'C-band (5.4 cm)': 0.054, 
                   'S-band (10.7 cm)': 0.107}
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 子图1: 多普勒频移 vs 速度（不同波长）
    ax1 = axes[0, 0]
    colors = {'X-band (3.2 cm)': 'red', 'C-band (5.4 cm)': 'green', 'S-band (10.7 cm)': 'blue'}
    for band, wl in wavelengths.items():
        f_d = calculate_doppler_frequency(v_range, wl)
        ax1.plot(v_range, f_d, color=colors[band], linewidth=2, label=band)
    ax1.set_xlabel('径向速度 (m/s)', fontsize=11)
    ax1.set_ylabel('多普勒频移 f_D (Hz)', fontsize=11)
    ax1.set_title('多普勒频移 vs 径向速度', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 子图2: 径向速度 vs 多普勒频移（反算）
    ax2 = axes[0, 1]
    f_d_range = np.linspace(-5000, 5000, 500)
    for band, wl in wavelengths.items():
        v = calculate_radial_velocity(f_d_range, wl)
        ax2.plot(f_d_range, v, color=colors[band], linewidth=2, label=band)
    ax2.set_xlabel('多普勒频移 f_D (Hz)', fontsize=11)
    ax2.set_ylabel('径向速度 (m/s)', fontsize=11)
    ax2.set_title('径向速度 vs 多普勒频移 (反算)', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 子图3: KDP vs 降雨率（不同波长）
    ax3 = axes[1, 0]
    R_range = np.linspace(0.1, 100, 100)
    for band, wl in wavelengths.items():
        kdp = [kdp_from_rainrate(R, wl) for R in R_range]
        ax3.plot(R_range, kdp, color=colors[band], linewidth=2, label=band)
    ax3.set_xlabel('降雨率 R (mm/h)', fontsize=11)
    ax3.set_ylabel('KDP (°/km)', fontsize=11)
    ax3.set_title('KDP vs 降雨率 (经验关系)', fontsize=12)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # 子图4: 差分相移 vs 距离（不同KDP）
    ax4 = axes[1, 1]
    r_range = np.linspace(0, 100, 100)
    kdp_values = [0.5, 1.0, 2.0, 5.0]
    for kdp in kdp_values:
        phidp = [calculate_differential_phase(kdp, r) for r in r_range]
        ax4.plot(r_range, phidp, linewidth=2, label=f'KDP={kdp}°/km')
    ax4.set_xlabel('距离 R (km)', fontsize=11)
    ax4.set_ylabel('ΦDP (°)', fontsize=11)
    ax4.set_title('差分相移 ΦDP 随距离累积', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_doppler_calculator()

## 3. 交互式计算器

In [ ]:
# 交互式计算器
def interactive_calculator(mode='velocity_to_freq', velocity=30.0, wavelength=0.10, 
                           f_d=600.0, distance=50.0, rainrate=20.0):
    """交互式多普勒和差分相移计算器"""
    
    if mode == 'velocity_to_freq':
        # 从速度计算频移
        f_d_calc = calculate_doppler_frequency(velocity, wavelength)
        print(f"\n=== 多普勒频移计算 ===")
        print(f"径向速度 v_r = {velocity:.1f} m/s")
        print(f"波长 λ = {wavelength*100:.1f} cm")
        print(f"多普勒频移 f_D = {f_d_calc:.1f} Hz")
        
    elif mode == 'freq_to_velocity':
        # 从频移计算速度
        v_calc = calculate_radial_velocity(f_d, wavelength)
        print(f"\n=== 径向速度计算 ===")
        print(f"多普勒频移 f_D = {f_d:.1f} Hz")
        print(f"波长 λ = {wavelength*100:.1f} cm")
        print(f"径向速度 v_r = {v_calc:.1f} m/s")
        
    elif mode == 'phidp_calculation':
        # 计算差分相移
        kdp = kdp_from_rainrate(rainrate, wavelength)
        phidp = calculate_differential_phase(kdp, distance)
        print(f"\n=== 差分相移计算 ===")
        print(f"降雨率 R = {rainrate:.1f} mm/h")
        print(f"波长 λ = {wavelength*100:.1f} cm")
        print(f"比差分相移 KDP = {kdp:.2f} °/km")
        print(f"距离 R = {distance:.1f} km")
        print(f"差分相移 ΦDP = {phidp:.1f} °")
    
    return None

# 创建交互式控件
style = {'description_width': '150px'}
layout = widgets.Layout(width='500px')

interact_calc = interact(interactive_calculator,
    mode=widgets.Dropdown(options=[('速度→频移', 'velocity_to_freq'), 
                                    ('频移→速度', 'freq_to_velocity'),
                                    ('ΦDP计算', 'phidp_calculation')],
                           value='velocity_to_freq', description='计算模式'),
    velocity=widgets.FloatText(value=30.0, description='径向速度 (m/s)', style=style, layout=layout),
    wavelength=widgets.FloatText(value=0.10, description='波长 (m)', style=style, layout=layout),
    f_d=widgets.FloatText(value=600.0, description='多普勒频移 (Hz)', style=style, layout=layout),
    distance=widgets.FloatText(value=50.0, description='距离 (km)', style=style, layout=layout),
    rainrate=widgets.FloatText(value=20.0, description='降雨率 (mm/h)', style=style, layout=layout)
)

## 4. 各波段典型值对照表

In [ ]:
def print_reference_table():
    """打印参考对照表"""
    print("\n=== 多普勒频移对照表 (v_r = 30 m/s) ===")
    print(f"{'波段':<15} {'波长 (cm)':<12} {'f_D (Hz)':<12}")
    print("-" * 40)
    for band, wl in [('X-band', 0.032), ('C-band', 0.054), ('S-band', 0.107)]:
        f_d = calculate_doppler_frequency(30.0, wl)
        print(f"{band:<15} {wl*100:<12.1f} {f_d:<12.1f}")
    
    print("\n=== KDP 对照表 (R = 10 mm/h) ===")
    print(f"{'波段':<15} {'KDP (°/km)':<12}")
    print("-" * 30)
    for band, wl in [('X-band', 0.032), ('C-band', 0.054), ('S-band', 0.107)]:
        kdp = kdp_from_rainrate(10.0, wl)
        print(f"{band:<15} {kdp:<12.2f}")
    
    print("\n=== ΦDP 累积示例 (KDP=1°/km, R=50km) ===")
    print(f"总差分相移 ΦDP = 50°")

In [ ]:
print_reference_table()

## 练习题

1. **频移计算**：S波段雷达探测到一个目标的多普勒频移为1000Hz，目标的径向速度是多少？

2. **KDP反演**：X波段雷达测量到某点的KDP为3°/km，估计当地的降雨率。

3. **ΦDP累积**：在C波段，KDP=2°/km的雨中传播80km后，总差分相移是多少？

4. **波段选择**：为什么S波段雷达的多普勒频移最小？这有什么优缺点？

## 参考文献

- Ryzhkov, A.V., and D.S. Zrnic, 2019: *Radar Polarimetry for Weather Observations*, Chapter 2, Springer.